# CSIRO Biomass - v2 Spatial-Aware Pooling Training

**主な改良点**:
1. AdaptiveAvgPool1d → SpatialAwarePooling（空間情報保持）
2. 物理制約付き損失関数
3. 軽量ステレオ融合

## 1. Setup & Installation

In [ ]:
# Update and install dependencies
!apt-get update -qq
!apt-get install -qq unzip

# Install libraries
!pip install -q --upgrade pip
!pip install -q --upgrade typing_extensions
!pip install -q timm==0.9.12
!pip install -q albumentations==1.3.1
!pip install -q pandas scikit-learn matplotlib tqdm
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118

In [ ]:
import os
import gc
import random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast
from torch.utils.checkpoint import checkpoint

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import r2_score
from tqdm.notebook import tqdm

import warnings
warnings.filterwarnings('ignore')

## 2. Configuration

In [ ]:
class CFG:
    # Paths
    DATA_DIR = Path("/workspace/data")
    OUTPUT_DIR = Path("/workspace/checkpoints_v2_spatial")
    
    # Model
    BACKBONE = "vit_huge_plus_patch16_dinov3.lvd1689m"
    PRETRAINED = True
    
    # Training - Memory optimized
    IMG_SIZES = [384, 448, 512]  # Multi-scale training
    BASE_IMG_SIZE = 448
    BATCH_SIZE = 1  # 24GB GPU用
    GRAD_ACC = 8    # 実効バッチサイズ = 8
    EPOCHS = 35
    LR = 1e-4
    MIN_LR = 1e-6
    WEIGHT_DECAY = 0.01
    
    # Augmentation
    AUG_PROB = 0.5
    MIXUP_ALPHA = 0.4
    
    # EMA & SWA
    USE_EMA = True
    EMA_DECAY = 0.995
    USE_SWA = True
    SWA_START_EPOCH = 25
    
    # Training settings
    N_FOLDS = 5
    SEED = 42
    NUM_WORKERS = 0  # Jupyter対応
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Target columns
    TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
    
# Create output directory
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CFG.DATA_DIR.mkdir(parents=True, exist_ok=True)

# Set seed
random.seed(CFG.SEED)
np.random.seed(CFG.SEED)
torch.manual_seed(CFG.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.SEED)

print(f"Device: {CFG.DEVICE}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu} ({vram:.1f} GB)")

## 3. Download and Prepare Data

In [ ]:
# Check if data exists
if not (CFG.DATA_DIR / "train.csv").exists():
    print("Downloading data...")
    !pip install -q kaggle
    
    # Setup Kaggle API credentials — never hardcode them in this notebook.
    # Option 1: set the KAGGLE_USERNAME / KAGGLE_KEY environment variables
    # Option 2: place your kaggle.json at ~/.kaggle/kaggle.json (chmod 600) beforehand
    has_env = bool(os.environ.get("KAGGLE_USERNAME")) and bool(os.environ.get("KAGGLE_KEY"))
    has_file = (Path.home() / ".kaggle" / "kaggle.json").exists()
    if not (has_env or has_file):
        raise RuntimeError(
            "Kaggle API credentials not found. Set the KAGGLE_USERNAME / KAGGLE_KEY "
            "environment variables, or upload kaggle.json to ~/.kaggle/ before running."
        )
    
    # Download competition data
    !kaggle competitions download -c csiro-biomass -p /workspace/data
    
    # Unzip
    import zipfile
    for file in CFG.DATA_DIR.glob("*.zip"):
        with zipfile.ZipFile(file, 'r') as z:
            z.extractall(CFG.DATA_DIR)

# Load data
train_df = pd.read_csv(CFG.DATA_DIR / "train.csv")
print(f"Train samples: {len(train_df)}")
print(f"Unique images: {train_df['image_path'].nunique()}")

## 4. 改良版Model（SpatialAwarePooling実装）

In [ ]:
class SpatialAwarePooling(nn.Module):
    """空間認識プーリング - AdaptiveAvgPool1dの改良版"""
    def __init__(self, dim=1280):
        super().__init__()
        # 各空間位置の重要度を学習
        self.attention = nn.Sequential(
            nn.Linear(dim, dim // 4),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(dim // 4, 1)
        )
        
        # 空間情報を保持する追加特徴
        self.spatial_features = nn.Sequential(
            nn.Conv1d(dim, dim // 2, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv1d(dim // 2, dim // 4, kernel_size=1)
        )
        
    def forward(self, x):
        # x: (batch, seq_len, dim)
        batch_size = x.shape[0]
        
        # 重要度計算
        attn_weights = self.attention(x)  # (batch, seq_len, 1)
        attn_weights = torch.softmax(attn_weights, dim=1)
        
        # 重み付き平均（重要な領域により注目）
        weighted_mean = torch.sum(x * attn_weights, dim=1)  # (batch, dim)
        
        # 空間的特徴の抽出
        x_t = x.transpose(1, 2)  # (batch, dim, seq_len)
        spatial_feat = self.spatial_features(x_t)  # (batch, dim//4, seq_len)
        
        # 最大値と平均値を組み合わせ（空間情報保持）
        spatial_max = torch.max(spatial_feat, dim=2)[0]  # (batch, dim//4)
        spatial_avg = torch.mean(spatial_feat, dim=2)    # (batch, dim//4)
        
        # 統合
        combined = torch.cat([
            weighted_mean,                              # (batch, dim)
            spatial_max,                                # (batch, dim//4)
            spatial_avg                                 # (batch, dim//4)
        ], dim=1)  # (batch, dim + dim//2)
        
        return combined

In [ ]:
class ImprovedLocalMambaBlock(nn.Module):
    """改良版LocalMambaBlock"""
    def __init__(self, dim, kernel_size=5, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        
        # Depthwise-Separable Convolution
        self.dwconv = nn.Conv1d(dim, dim, kernel_size, padding=kernel_size//2, groups=dim)
        self.pwconv = nn.Conv1d(dim, dim, 1)
        
        # チャネル注意機構
        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(dim, dim//16, 1),
            nn.ReLU(),
            nn.Conv1d(dim//16, dim, 1),
            nn.Sigmoid()
        )
        
        self.gate = nn.Linear(dim, dim)
        self.proj = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        shortcut = x
        x = self.norm(x)
        
        # ゲート制御
        x = x * torch.sigmoid(self.gate(x))
        
        # 改良畳み込み
        x_t = x.transpose(1, 2)
        x_t = self.dwconv(x_t)
        x_t = self.pwconv(x_t)
        
        # チャネル注意
        attn = self.channel_attn(x_t)
        x_t = x_t * attn
        
        x = x_t.transpose(1, 2)
        x = self.proj(x)
        
        return shortcut + self.drop(x)

In [ ]:
class LightweightStereoFusion(nn.Module):
    """軽量ステレオ融合"""
    def __init__(self, dim=1280):
        super().__init__()
        # 軽量クロス接続
        self.cross_proj = nn.Linear(dim, dim // 4)
        self.gate = nn.Sequential(
            nn.Linear(dim // 4, dim // 4),
            nn.Sigmoid()
        )
        self.expand = nn.Linear(dim // 4, dim)
        
    def forward(self, left_feat, right_feat):
        # 軽量クロス情報交換
        left_cross = self.cross_proj(right_feat)   # 右→左の情報
        right_cross = self.cross_proj(left_feat)   # 左→右の情報
        
        # ゲート制御で選択的融合
        left_gate = self.gate(left_cross)
        right_gate = self.gate(right_cross)
        
        # 拡張して元の次元に戻す
        left_enhanced = left_feat + self.expand(left_gate * left_cross)
        right_enhanced = right_feat + self.expand(right_gate * right_cross)
        
        return torch.cat([left_enhanced, right_enhanced], dim=1)

In [ ]:
class ImprovedBiomassModel(nn.Module):
    """v2: 空間認識プーリング版モデル"""
    def __init__(self, model_name=CFG.BACKBONE, pretrained=True):
        super().__init__()
        # Backbone
        self.backbone = timm.create_model(
            model_name, 
            pretrained=pretrained, 
            num_classes=0, 
            global_pool=""
        )
        
        # Enable gradient checkpointing for memory efficiency
        if hasattr(self.backbone, 'set_grad_checkpointing'):
            self.backbone.set_grad_checkpointing(True)
        
        nf = self.backbone.num_features  # 1280
        
        # 軽量ステレオ融合
        self.stereo_fusion = LightweightStereoFusion(nf)
        
        # 改良Mamba融合
        self.fusion = nn.Sequential(
            ImprovedLocalMambaBlock(nf, kernel_size=5, dropout=0.1),
            ImprovedLocalMambaBlock(nf, kernel_size=5, dropout=0.1)
        )
        
        # 空間認識プーリング（AdaptiveAvgPool1dの代替）
        self.pool = SpatialAwarePooling(nf)
        pool_output_dim = nf + nf // 2  # 1280 + 640 = 1920
        
        # ヘッド（pool出力次元に対応）
        self.head_green = nn.Sequential(
            nn.Linear(pool_output_dim, nf//2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(nf//2, 1),
            nn.Softplus()
        )
        
        self.head_dead = nn.Sequential(
            nn.Linear(pool_output_dim, nf//2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(nf//2, 1),
            nn.Softplus()
        )
        
        self.head_clover = nn.Sequential(
            nn.Linear(pool_output_dim, nf//2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(nf//2, 1),
            nn.Softplus()
        )

    def forward(self, x):
        left, right = x
        
        # Backbone
        x_l = self.backbone(left)    # (batch, 784, 1280)
        x_r = self.backbone(right)   # (batch, 784, 1280)
        
        # ステレオ融合
        x = self.stereo_fusion(x_l, x_r)  # (batch, 1568, 1280)
        
        # Mamba融合
        x = self.fusion(x)  # (batch, 1568, 1280)
        
        # 空間認識プーリング
        x = self.pool(x)  # (batch, 1920)
        
        # マルチタスク予測
        green = self.head_green(x)
        dead = self.head_dead(x)
        clover = self.head_clover(x)
        
        # 物理制約
        gdm = green + clover
        total = green + clover + dead
        
        return torch.cat([green, dead, clover, gdm, total], dim=1)

## 5. 物理制約付き損失関数

In [ ]:
class PhysicsConstrainedLoss(nn.Module):
    """物理制約付き損失関数"""
    def __init__(self):
        super().__init__()
        # 各ターゲットの不確実性重み（学習可能）
        self.log_vars = nn.Parameter(torch.zeros(5))
        
    def forward(self, pred, target):
        # 基本MSE損失
        mse_loss = F.mse_loss(pred, target, reduction='none')
        
        # 物理制約違反ペナルティ
        # GDM = Green + Clover
        gdm_violation = F.relu(torch.abs(pred[:, 3] - (pred[:, 0] + pred[:, 2])) - 0.01)
        
        # Total = Green + Dead + Clover  
        total_violation = F.relu(torch.abs(pred[:, 4] - (pred[:, 0] + pred[:, 1] + pred[:, 2])) - 0.01)
        
        # 非負制約
        negative_penalty = F.relu(-pred).mean()
        
        # 不確実性重み付き損失
        precision = torch.exp(-self.log_vars)
        weighted_mse = torch.sum(precision * mse_loss + self.log_vars, dim=1)
        
        # 総損失
        total_loss = weighted_mse.mean() + \
                    0.1 * gdm_violation.mean() + \
                    0.1 * total_violation.mean() + \
                    0.05 * negative_penalty
        
        return total_loss

## 6. Dataset & Augmentation

In [ ]:
class BiomassDataset(Dataset):
    def __init__(self, df, data_dir, transform=None, is_train=True):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.transform = transform
        self.is_train = is_train
        self.targets = CFG.TARGETS
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load image
        img_path = self.data_dir / row['image_path']
        img = Image.open(img_path).convert('RGB')
        
        # Split into left and right
        w, h = img.size
        left = img.crop((0, 0, w // 2, h))
        right = img.crop((w // 2, 0, w, h))
        
        # Convert to array
        left = np.array(left)
        right = np.array(right)
        
        # Apply transforms
        if self.transform:
            # 同じ変換を両方に適用
            augmented = self.transform(image=left)
            left = augmented['image']
            
            augmented = self.transform(image=right)
            right = augmented['image']
        
        # Get targets
        if self.is_train:
            targets = torch.tensor([row[t] for t in self.targets], dtype=torch.float32)
            return left, right, targets
        else:
            return left, right

def get_transforms(img_size):
    """データ拡張の定義"""
    train_transform = A.Compose([
        A.RandomResizedCrop(img_size, img_size, scale=(0.8, 1.0)),
        A.OneOf([
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=1),
            A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=1),
            A.CLAHE(clip_limit=2.0, p=1),
        ], p=CFG.AUG_PROB),
        A.OneOf([
            A.GaussNoise(var_limit=(10, 50), p=1),
            A.GaussianBlur(blur_limit=3, p=1),
        ], p=0.2),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])
    
    val_transform = A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])
    
    return train_transform, val_transform

## 7. Training Loop

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scaler, device, img_size):
    model.train()
    losses = []
    
    pbar = tqdm(loader, desc='Training')
    for batch_idx, (left, right, targets) in enumerate(pbar):
        left = left.to(device)
        right = right.to(device)
        targets = targets.to(device)
        
        # Mixed precision training
        with autocast():
            outputs = model((left, right))
            loss = criterion(outputs, targets)
            loss = loss / CFG.GRAD_ACC
        
        scaler.scale(loss).backward()
        
        # Gradient accumulation
        if (batch_idx + 1) % CFG.GRAD_ACC == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        losses.append(loss.item() * CFG.GRAD_ACC)
        pbar.set_postfix({'loss': np.mean(losses)})
        
        # Memory cleanup
        if batch_idx % 20 == 0:
            torch.cuda.empty_cache()
    
    return np.mean(losses)

def validate(model, loader, criterion, device):
    model.eval()
    losses = []
    predictions = []
    targets_list = []
    
    with torch.no_grad():
        pbar = tqdm(loader, desc='Validation')
        for left, right, targets in pbar:
            left = left.to(device)
            right = right.to(device)
            targets = targets.to(device)
            
            with autocast():
                outputs = model((left, right))
                loss = criterion(outputs, targets)
            
            losses.append(loss.item())
            predictions.append(outputs.cpu())
            targets_list.append(targets.cpu())
    
    predictions = torch.cat(predictions)
    targets = torch.cat(targets_list)
    
    # Calculate R2 scores
    r2_scores = []
    for i in range(len(CFG.TARGETS)):
        r2 = r2_score(targets[:, i], predictions[:, i])
        r2_scores.append(r2)
    
    return np.mean(losses), np.mean(r2_scores), r2_scores

## 8. Training Main

In [ ]:
def train_fold(fold, train_idx, val_idx):
    print(f"\n{'='*50}")
    print(f"Fold {fold}")
    print(f"{'='*50}")
    
    # Prepare data
    train_fold = train_wide.iloc[train_idx]
    val_fold = train_wide.iloc[val_idx]
    
    # Create model
    model = ImprovedBiomassModel(CFG.BACKBONE, pretrained=CFG.PRETRAINED)
    model = model.to(CFG.DEVICE)
    
    # Optimizer & Scheduler
    optimizer = AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS, eta_min=CFG.MIN_LR)
    
    # Loss function
    criterion = PhysicsConstrainedLoss().to(CFG.DEVICE)
    
    # Mixed precision
    scaler = GradScaler()
    
    # EMA
    if CFG.USE_EMA:
        from copy import deepcopy
        ema_model = deepcopy(model)
        ema_model.eval()
    
    # SWA
    if CFG.USE_SWA:
        swa_model = torch.optim.swa_utils.AveragedModel(model)
    
    best_score = -float('inf')
    
    for epoch in range(CFG.EPOCHS):
        print(f"\nEpoch {epoch+1}/{CFG.EPOCHS}")
        
        # Multi-scale training
        if epoch < 10:
            img_size = CFG.IMG_SIZES[0]
        elif epoch < 20:
            img_size = CFG.IMG_SIZES[1]
        else:
            img_size = CFG.IMG_SIZES[2]
        
        print(f"Image size: {img_size}")
        
        # Get transforms
        train_transform, val_transform = get_transforms(img_size)
        
        # Create datasets
        train_dataset = BiomassDataset(train_fold, CFG.DATA_DIR, train_transform, is_train=True)
        val_dataset = BiomassDataset(val_fold, CFG.DATA_DIR, val_transform, is_train=True)
        
        # Create loaders
        train_loader = DataLoader(
            train_dataset, batch_size=CFG.BATCH_SIZE, shuffle=True, 
            num_workers=CFG.NUM_WORKERS, pin_memory=True
        )
        
        val_loader = DataLoader(
            val_dataset, batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
            num_workers=CFG.NUM_WORKERS, pin_memory=True
        )
        
        # Train
        train_loss = train_epoch(model, train_loader, criterion, optimizer, scaler, CFG.DEVICE, img_size)
        
        # Validate
        val_loss, val_score, val_r2_scores = validate(model, val_loader, criterion, CFG.DEVICE)
        
        # Update EMA
        if CFG.USE_EMA:
            for ema_param, model_param in zip(ema_model.parameters(), model.parameters()):
                ema_param.data.mul_(CFG.EMA_DECAY).add_(model_param.data, alpha=1 - CFG.EMA_DECAY)
        
        # Update SWA
        if CFG.USE_SWA and epoch >= CFG.SWA_START_EPOCH:
            swa_model.update_parameters(model)
        
        # Scheduler step
        scheduler.step()
        
        # Print metrics
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Val Loss: {val_loss:.4f}")
        print(f"Val R2 Score: {val_score:.4f}")
        print(f"R2 by target: {dict(zip(CFG.TARGETS, val_r2_scores))}")
        
        # Save best model
        if val_score > best_score:
            best_score = val_score
            torch.save(model.state_dict(), CFG.OUTPUT_DIR / f"best_fold{fold}.pth")
            if CFG.USE_EMA:
                torch.save(ema_model.state_dict(), CFG.OUTPUT_DIR / f"best_ema_fold{fold}.pth")
            print(f"✅ Saved best model (R2: {best_score:.4f})")
    
    # Save SWA model
    if CFG.USE_SWA:
        torch.save(swa_model.module.state_dict(), CFG.OUTPUT_DIR / f"best_swa_fold{fold}.pth")
    
    # Cleanup
    del model, optimizer, scheduler
    if CFG.USE_EMA:
        del ema_model
    if CFG.USE_SWA:
        del swa_model
    torch.cuda.empty_cache()
    gc.collect()
    
    return best_score

In [ ]:
# Prepare data
train_wide = train_df.groupby('image_path').agg({
    'Dry_Green_g': 'mean',
    'Dry_Dead_g': 'mean', 
    'Dry_Clover_g': 'mean',
    'GDM_g': 'mean',
    'Dry_Total_g': 'mean',
    'site': 'first'
}).reset_index()

# Create stratified folds
train_wide['bins'] = pd.qcut(train_wide['Dry_Total_g'], q=10, labels=False, duplicates='drop')
sgkf = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)

# Train all folds
scores = []
for fold, (train_idx, val_idx) in enumerate(sgkf.split(train_wide, train_wide['bins'], groups=train_wide['site'])):
    score = train_fold(fold, train_idx, val_idx)
    scores.append(score)

print(f"\n{'='*50}")
print(f"Cross Validation Results")
print(f"{'='*50}")
for fold, score in enumerate(scores):
    print(f"Fold {fold}: R2 = {score:.4f}")
print(f"Mean R2: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

## 9. Summary

### 改良点の効果
1. **SpatialAwarePooling**: 空間情報を保持し、重要領域に注目
2. **ImprovedLocalMambaBlock**: チャネル注意とDepthwise-Separable Conv
3. **LightweightStereoFusion**: 左右画像の相互作用を学習
4. **PhysicsConstrainedLoss**: 物理制約を損失関数に組み込み

### 期待される性能向上
- AdaptiveAvgPool1d修正: +5-8%
- 物理制約損失: +3-5%  
- ステレオ融合改良: +2-4%
- **合計: +10-17%** (R² 0.93-1.02)